In [7]:
import importlib, collectri_ctx as cc
importlib.reload(cc)

# Use `decoupleR::get_collectri(organism='human', split_complexes=FALSE)` if needed.
COLLECTRI_CSV = "NEEDS YOUR PATH/collectri_human_net.csv"
PUBMED_CACHE = "./pubmed_cache"  # same path you pass to the loader

from Bio import Entrez
Entrez.email = "REDACTED"
Entrez.api_key = "REDACTED"

pmids = cc.collect_unique_pmids(COLLECTRI_CSV)
print("Unique PMIDs:", len(pmids))
cc.prefetch_pubmed_to_cache(pmids, PUBMED_CACHE, batch_size=200, delay=0.12)


Unique PMIDs: 38028
Prefetching 36843 PubMed records in batches of 200...


efetch batches: 100%|█████████████████████████████████████████████████████████████████| 185/185 [04:56<00:00,  1.60s/it]


In [8]:
rec = cc.load_cached_pubmed("10666449", PUBMED_CACHE)  # one of your PMIDs
print(rec["title"][:120])
print(rec["mesh"][:8])

An 'environment to nucleus' signaling system operates in B lymphocytes: redox status modulates BSAP/Pax-5 activation thr
['b-lymphocytes', 'biological transport', 'carbon-oxygen lyases', 'cell line', 'dna-(apurinic or apyrimidinic site) lyase', 'dna-binding proteins', 'humans', 'nuclear proteins']


In [9]:
# Rebuild docs FROM CACHE ONLY
import importlib, collectri_ctx as cc
importlib.reload(cc)

# Use `decoupleR::get_collectri(organism='human', split_complexes=FALSE)` if needed.
COLLECTRI_CSV = "NEEDS YOUR PATH/collectri_human_net.csv"
PUBMED_CACHE = "./pubmed_cache"

docs, ctx_hist = cc.load_collectri_with_context_v2(
    COLLECTRI_CSV,
    source_name="CollectRI-Human",
    ctx_thresh=0.3,   # you can try 0.2 if you want to be more inclusive
    top_n=2,
    cache_dir=PUBMED_CACHE
)

print(f"Built {len(docs)} docs")
for k, v in ctx_hist.most_common(10):
    print(f"{k:10s}: {v}")


Build docs: 100%|███████████████████████████████████████████████████████████████| 43178/43178 [00:09<00:00, 4729.51it/s]

Built 92703 docs
immune    : 27930
general   : 26083
epithelial: 8369
hepatic   : 7821
neural    : 7450
stem      : 6221
muscle    : 3785
renal     : 3247
cardiac   : 1797


In [13]:
rows = cc.get_all_collectri_entries(
    docs, "PAX5", "CD19",
    context="B cells",
    soft_threshold=0.3,
    strict_context=True,              # hide non-immune rows
    show_nonmatching_fallback=False   # no fallback noise
)
cc.print_collectri_rows(rows, "B cells")


1. w=1.00 ctx=immune conf=1.00 [✓] reg=activation pmids=10666449
    TF PAX5 regulates CD19 by activation (weight=1.0) in immune context (conf=1.00). PMID: 10666449. Source: CollectRI-Human. 

2. w=1.00 ctx=immune conf=1.00 [✓] reg=activation pmids=12907641
    TF PAX5 regulates CD19 by activation (weight=1.0) in immune context (conf=1.00). PMID: 12907641. Source: CollectRI-Human. 

3. w=1.00 ctx=immune conf=1.00 [✓] reg=activation pmids=1375324
    TF PAX5 regulates CD19 by activation (weight=1.0) in immune context (conf=1.00). PMID: 1375324. Source: CollectRI-Human. 

4. w=1.00 ctx=immune conf=1.00 [✓] reg=activation pmids=15163413
    TF PAX5 regulates CD19 by activation (weight=1.0) in immune context (conf=1.00). PMID: 15163413. Source: CollectRI-Human. 

5. w=1.00 ctx=immune conf=1.00 [✓] reg=activation pmids=17513763
    TF PAX5 regulates CD19 by activation (weight=1.0) in immune context (conf=1.00). PMID: 17513763. Source: CollectRI-Human. 

6. w=1.00 ctx=immune conf=1.00 [✓] re

In [14]:
import pickle

# Save
with open("collectri_docs.pkl", "wb") as f:
    pickle.dump(docs, f)

with open("collectri_ctx_hist.pkl", "wb") as f:
    pickle.dump(ctx_hist, f)